# Scraping de coches de segunda mano 

### Que queremos 
- hacer scraping de paginas web de coches de segunda mano
- 
- utilizar rag para guardar los precios de los coches


### login en el modelo 

In [24]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')

MODEL = "gemini-2.0-flash"
openai = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta", api_key=api_key)

response = openai.chat.completions.create(
 model=MODEL,
 messages=[{"role": "user", "content": "¿Cuánto son 2 + 2?"}]
)

print(response.choices[0].message.content)

BadRequestError: Error code: 400 - [{'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}]

In [9]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import time

def scrape_autofesa_final():
    """Scraper definitivo con el selector de paginación correcto"""
    options = Options()
    # options.add_argument('--headless')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
    
    driver = webdriver.Chrome(options=options)
    resultados = []
    
    try:
        url = "https://www.autofesa.com/coches-segunda-mano"
        print(f"🔗 Accediendo a: {url}")
        driver.get(url)
        
        wait = WebDriverWait(driver, 10)

        # ==== MANEJO DE COOKIES ====
        try:
            print("🍪 Buscando banner de cookies...")
            cookie_button = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
            )
            cookie_button.click()
            print("   ✓ Cookies aceptadas por ID.")
            time.sleep(1) 
        except TimeoutException:
            try:
                # Fallback por si el ID cambia, buscamos por texto
                cookie_button_text = WebDriverWait(driver, 2).until(
                    EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Aceptar')]")) # Buscamos un botón que diga Aceptar
                )
                cookie_button_text.click()
                print("   ✓ Cookies aceptadas por Texto.")
                time.sleep(1)
            except TimeoutException:
                print("   ℹ️ No se encontró banner de cookies o ya estaba aceptado.")
        # ===================================
        
        # ==== LÓGICA DE PAGINACIÓN ====
        page_num = 1
        while True:
            print(f"\n📄 Empezando a scrapear página: {page_num}")
            try:
                # 1. Esperar a que los coches de ESTA página carguen
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".vehicle-list__item")))
                time.sleep(1) 

                # 2. Scrapear los coches de la página ACTUAL
                car_elements = driver.find_elements(By.CSS_SELECTOR, ".vehicle-list__item")
                print(f"🚗 Encontrados {len(car_elements)} coches en esta página.")
                
                for car in car_elements:
                    idx = len(resultados) + 1 
                    try:
                        # Título
                        try:
                            title_elem = car.find_element(By.CSS_SELECTOR, ".vehicle-card__title")
                            title = title_elem.text.strip()
                        except:
                            title = "Sin título"
                        
                        # Precio
                        price = "Precio no disponible"
                        price_selectors = [
                            ".vehicle-card__price",
                            ".vehicle-card__price--soldout",
                            ".vehicle-card__price span"
                        ]
                        for sel in price_selectors:
                            try:
                                price_elem = car.find_element(By.CSS_SELECTOR, sel)
                                if price_elem.text.strip():
                                    price = price_elem.text.strip()
                                    break
                            except:
                                continue
                        
                        # Link
                        try:
                            link_elem = car.find_element(By.CSS_SELECTOR, "a")
                            link = link_elem.get_attribute("href")
                        except:
                            link = "Sin enlace"
                        
                        # Información adicional
                        try:
                            features = car.find_elements(By.CSS_SELECTOR, ".vehicle-card__features .list .item")
                            additional_info = " | ".join([f.text.strip() for f in features if f.text.strip()])
                        except:
                            additional_info = "Sin información adicional"
                        
                        resultados.append({
                            "#": idx,
                            "Modelo": title,
                            "Precio": price,
                            "Información": additional_info,
                            "Link": link
                        })
                        
                        if idx % 10 == 0: 
                            print(f"   ✓ Coche {idx}: {title[:30]}... - {price}")
                            
                    except Exception as e:
                        print(f"   ❌ Error procesando coche {idx}: {e}")
                        continue
                
                
                # 3. Buscar el botón "Siguiente" y hacer clic

                # Hacemos scroll hasta el final para asegurar que el botón es visible
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(0.5) 

                print("Buscando botón 'Siguiente'...")
                # ==== AQUÍ ESTÁ LA CORRECCIÓN ====
                # Usamos 'title="Página siguiente"' que es el texto exacto del HTML
                next_button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.page-link[title="Página siguiente"]')))
                
                print("Haciendo clic en 'Siguiente'...")
                driver.execute_script("arguments[0].click();", next_button)
                page_num += 1
                
                time.sleep(2) # Espera a que la nueva página cargue

            except TimeoutException:
                # 4. Si no se encuentra el botón "Siguiente" (da Timeout)
                
                print("\n🏁 No se encontró más botón 'Siguiente'. Es la última página.")
                try:
                    driver.save_screenshot("debug_final_page.png")
                    print(f"   📸 Se ha guardado un pantallazo de la última página como 'debug_final_page.png'.")
                except Exception as e:
                    print(f"   No se pudo guardar el pantallazo: {e}")

                break # Salir del bucle 'while True'
            
            except Exception as e:
                print(f"❌ Error inesperado en el bucle de paginación: {e}")
                driver.save_screenshot("debug_error_inesperado.png")
                print("   📸 Pantallazo de error guardado como 'debug_error_inesperado.png'.")
                break 
    
    except Exception as e:
        print(f"❌ Error general: {e}")
    
    finally:
        driver.quit()
    
    return resultados

# Ejecutar scraper
print("🚀 Iniciando scraper con selector corregido...")
resultados_final = scrape_autofesa_final()
print(f"\n✅ Scraping completado. Total: {len(resultados_final)} coches extraídos.")

🚀 Iniciando scraper con selector corregido...
🔗 Accediendo a: https://www.autofesa.com/coches-segunda-mano
🍪 Buscando banner de cookies...
   ℹ️ No se encontró banner de cookies o ya estaba aceptado.

📄 Empezando a scrapear página: 1
🚗 Encontrados 30 coches en esta página.
   ✓ Coche 10: Aston Martin DBS DBS COUPE CAS... - 149.500
€
OFERTA
   ✓ Coche 20: Audi A5 Cabrio 2.0 TFSI MHEV 1... - 34.650 €
31.650
€
OFERTA
   ✓ Coche 30: Audi Q2 30 TFSI DESIGN 115CV 5... - 20.850
€
OFERTA
Buscando botón 'Siguiente'...
Haciendo clic en 'Siguiente'...

📄 Empezando a scrapear página: 2
🚗 Encontrados 30 coches en esta página.
   ✓ Coche 40: Bentley Continental GT 4.0 V8 ... - 229.500 €
219.850
€
OFERTA
   ✓ Coche 50: BMW 318 2.0 TOURING AUTO 143cv... - 13.650 €
11.950
€
OFERTA
   ✓ Coche 60: BMW 530 2.0 iPERFORMANCE AUTO ... - 28.500 €
26.500
€
OFERTA
Buscando botón 'Siguiente'...
Haciendo clic en 'Siguiente'...

📄 Empezando a scrapear página: 3
🚗 Encontrados 30 coches en esta página.
   ✓ Coche 70

### Diseño final del scraping 

In [10]:
import pandas as pd


# Convertir lista de diccionarios a DataFrame
df = pd.DataFrame(resultados_final)

# Mostrar toda la tabla en consola
pd.set_option('display.max_rows', None)   # Mostrar todas las filas
pd.set_option('display.max_columns', None)  # Mostrar todas las columnas
pd.set_option('display.width', 1000)  # Evitar que se corte horizontalmente

print(df)


       #                                             Modelo                         Precio                                        Información                                               Link
0      1  Abarth 500 1.4T JET 595 165CV 70TH ANNIVERSARY...              17.350\n€\nOFERTA  2020 | 83.900km | Gasolina | Manual | Utilitar...  https://www.autofesa.com/coches-de-ocasion/aba...
1      2                          Aixam S10 SPORT S10 SPORT           Precio no disponible  2024 | 3.274km | Diesel | Manual | Utilitario ...  https://www.autofesa.com/coches-de-ocasion/aix...
2      3        Aixam S8 COUPE S8 COUPE 9CV AUTO 3P # CUERO              10.450\n€\nOFERTA  2015 | 44.144km | Diesel | Automático | Utilit...  https://www.autofesa.com/coches-de-ocasion/aix...
3      4  Alfa Romeo Giulietta GIULIETTA 1.4T 170CV LUSS...              16.850\n€\nOFERTA  2018 | 39.200km | Gasolina | Automático | Util...  https://www.autofesa.com/coches-de-ocasion/alf...
4      5  Alfa Romeo Giulietta GIUL

In [12]:
import chromadb
from sentence_transformers import SentenceTransformer
import time

# --- VALIDACIÓN ---
# Asegúrate de que 'resultados_final' existe por tu scraping anterior
if 'resultados_final' not in locals() or not resultados_final:
    print("❌ ERROR: No se encontró la variable 'resultados_final'.")
    print("Asegúrate de ejecutar tu script de scraping en una celda anterior.")
    # (Pongo datos de ejemplo para que puedas probar sin el scraper)
    resultados_final = [
        {'#': 1, 'Modelo': 'SEAT Arona 1.0 TSI 85kW (115CV) St&Sp FR', 'Precio': '16.890 €', 'Información': '2019 | 63.811 km | Gasolina', 'Link': 'https_link1'},
        {'#': 2, 'Modelo': 'AUDI A3 Sportback 2.0 TDI 110kW (150CV) S tronic', 'Precio': '22.490 €', 'Información': '2020 | 88.028 km | Diésel', 'Link': 'https...'},
        {'#': 3, 'Modelo': 'BMW Serie 1 118d 110kW (150CV) Aut.', 'Precio': '24.990 €', 'Información': '2020 | 79.418 km | Diésel', 'Link': 'https_link3'},
    ]
    print("ℹ️ Usando datos de ejemplo para continuar...")
else:
    print(f"🚗 ¡Éxito! Encontrados {len(resultados_final)} coches de tu scraping.")


# --- 1. PREPARAR DOCUMENTOS ---
documentos = []
metadatos = []  # El nombre de esta variable está bien, es solo una lista
ids = []

print("📄 Preparando documentos de texto...")
for coche in resultados_final:
    texto_documento = f"Modelo: {coche['Modelo']}. Precio: {coche['Precio']}. Detalles: {coche['Información']}."
    documentos.append(texto_documento)
    
    metadatos.append({
        "modelo": coche['Modelo'],
        "precio": coche['Precio'],
        "link": coche['Link'],
        "info": coche['Información']
    })
    
    ids.append(f"coche_{coche['#']}")

# --- 2. INICIAR LA BASE DE DATOS VECTORIAL (ChromaDB) ---
client = chromadb.PersistentClient(path="./db_coches")
collection = client.get_or_create_collection(name="coches_autofesa")
print("🗂️  Base de datos ChromaDB iniciada.")

# --- 3. VECTORIZAR ---
print("🧠 Cargando modelo de embeddings (all-MiniLM-L6-v2)...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("⏳ Empezando vectorización... Esto puede tardar unos minutos.")
start_time = time.time()
embeddings = embedding_model.encode(documentos)
end_time = time.time()
print(f"   ✓ Vectorización completada en {end_time - start_time:.2f} segundos.")

# --- 4. AÑADIR A LA BASE DE DATOS (CON CORRECCIÓN) ---
print("💾 Añadiendo vectores a ChromaDB...")
collection.upsert(
    embeddings=embeddings.tolist(),
    documents=documentos,
    metadatas=metadatos,  # <-- ¡CORREGIDO!
    ids=ids
)

print("\n✅ ¡Éxito! Base de datos vectorial creada y poblada.")

# --- 5. PRUEBA DE BÚSQUEDA RÁPIDA ---
print("\n--- Prueba de Búsqueda Vectorial ---")
pregunta_test = "Busco un Audi diésel"
print(f"Pregunta: '{pregunta_test}'")

pregunta_embedding = embedding_model.encode([pregunta_test]).tolist()
results = collection.query(
    query_embeddings=pregunta_embedding,
    n_results=3
)

print("\nResultados más similares (datos crudos):")
print(results['documents'])

🚗 ¡Éxito! Encontrados 697 coches de tu scraping.
📄 Preparando documentos de texto...
🗂️  Base de datos ChromaDB iniciada.
🧠 Cargando modelo de embeddings (all-MiniLM-L6-v2)...
⏳ Empezando vectorización... Esto puede tardar unos minutos.
   ✓ Vectorización completada en 3.58 segundos.
💾 Añadiendo vectores a ChromaDB...

✅ ¡Éxito! Base de datos vectorial creada y poblada.

--- Prueba de Búsqueda Vectorial ---
Pregunta: 'Busco un Audi diésel'

Resultados más similares (datos crudos):
[['Modelo: Audi A6 2.0TDI 177 AUTO. Precio: Precio no disponible. Detalles: 2013 | 124.707km | Diesel | Automático | Berlina | Novedad.', 'Modelo: Audi A6 Allroad A6 ALLROAD QUATTRO 3.0 TDI STRONIC 204CV 5P # CUERO,BIXENON,NAVY. Precio: 17.450\n€\nOFERTA. Detalles: 2013 | 159.000km | Diesel | Automático | Familiar | 479€/mes | Novedad.', 'Modelo: Audi A4 AVANT 2.0TDI AUTO 190cv 5P # NAVY, PARKTRONIC. Precio: 20.650\n€\nOFERTA. Detalles: 2018 | 100.800km | Diesel | Automático | Familiar | 315€/mes | Novedad.']

In [26]:
import os
import chromadb
import google.generativeai as genai
from sentence_transformers import SentenceTransformer

# --- 1. CONFIGURACIÓN API KEY ---
os.environ["GOOGLE_API_KEY"] = "TU_API_KEY_DE_GOOGLE_AQUI" 

if "TU_API_KEY" in os.environ["GOOGLE_API_KEY"]:
    print("🛑 ¡ALERTA! Reemplaza 'TU_API_KEY_DE_GOOGLE_AQUI' con tu API Key de Google AI Studio.")
else:
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
    print("   ✓ API Key de Google cargada.")

# --- 2. CARGAR COMPONENTES ---

# 🧠 Modelo de Embeddings
print("🧠 Cargando modelo de embeddings (all-MiniLM-L6-v2)...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# 🗂️ Conexión a la Base de Datos Vectorial
print("🗂️  Conectando a la base de datos ChromaDB existente...")
client = chromadb.PersistentClient(path="./db_coches")
collection = client.get_collection(name="coches_autofesa") 

# 🤖 Configurar el LLM (Gemini)
print("🤖 Configurando el LLM (Google Gemini)...")
# AQUÍ ESTÁ LA CORRECCIÓN:
llm = genai.GenerativeModel('gemini-2.0-flash')

print("\n✅ ¡Componentes listos para RAG manual!")

🛑 ¡ALERTA! Reemplaza 'TU_API_KEY_DE_GOOGLE_AQUI' con tu API Key de Google AI Studio.
🧠 Cargando modelo de embeddings (all-MiniLM-L6-v2)...
🗂️  Conectando a la base de datos ChromaDB existente...
🤖 Configurando el LLM (Google Gemini)...

✅ ¡Componentes listos para RAG manual!


In [27]:
# --- 1. Definir la plantilla del Prompt ---
# (La misma plantilla de antes)
template_prompt = """
Eres un asistente de ventas experto de Autofesa.
Tu trabajo es responder a la pregunta del cliente de forma amable y concisa.
Basa tu respuesta ÚNICAMENTE en la siguiente lista de coches encontrados en el inventario.
No inventes coches o detalles que no estén en la lista.
Si encuentras un coche relevante, incluye el modelo, precio e información.

Contexto (Coches encontrados):
{contexto}

Pregunta del cliente:
{pregunta}

Respuesta:
"""

# --- 2. Hacemos la pregunta ---
mi_pregunta = "Quiero un coche de gasolina por menos de 17.000 euros"
print(f"Pregunta: {mi_pregunta}\n")
print("Procesando RAG manualmente...")

# --- 3. El Flujo RAG Manual ---

# PASO 1: Vectorizar la pregunta del usuario
print("   (1/4) Vectorizando la pregunta...")
pregunta_embedding = embedding_model.encode([mi_pregunta]).tolist()

# PASO 2: Buscar en la Base de Datos Vectorial (ChromaDB)
print("   (2/4) Buscando en ChromaDB...")
results = collection.query(
    query_embeddings=pregunta_embedding,
    n_results=5  # Pedimos los 5 coches más relevantes
)
documentos_encontrados = results['documents'][0]

# PASO 3: Construir el Contexto y el Prompt
print("   (3/4) Construyendo el prompt final...")
# Unimos todos los documentos encontrados en un solo bloque de texto
contexto_str = "\n---\n".join(documentos_encontrados)

# Insertamos el contexto y la pregunta en la plantilla (con un f-string)
prompt_final = template_prompt.format(contexto=contexto_str, pregunta=mi_pregunta)

# (Opcional: puedes imprimir el 'prompt_final' para ver qué le envías al LLM)
# print("\n--- PROMPT ENVIADO A GEMINI ---\n", prompt_final)
# print("---------------------------------\n")

# PASO 4: Llamar al LLM (Gemini) con el prompt final
print("   (4/4) Enviando prompt al LLM (Gemini)...")
respuesta_llm = llm.generate_content(prompt_final)

# --- 4. Mostrar la respuesta ---
print("\n--- Respuesta de Gemini ---")
print(respuesta_llm.text)

Pregunta: Quiero un coche de gasolina por menos de 17.000 euros

Procesando RAG manualmente...
   (1/4) Vectorizando la pregunta...
   (2/4) Buscando en ChromaDB...
   (3/4) Construyendo el prompt final...
   (4/4) Enviando prompt al LLM (Gemini)...

--- Respuesta de Gemini ---
¡Claro! Tenemos varias opciones de gasolina por debajo de 17.000€:

*   **Opel Karl 1.0 ROCKS 73CV 5P:** Por 9.500€, es un utilitario de 2018 con 46.217km.
*   **Fiat Punto 1.2 POP 69CV 5P:** Por 7.850€, es un utilitario de 2015 con 34.000km.
*   **Fiat Bravo 1.4i 90CV 5P:** Por 5.750€, es un utilitario de 2013 con 150.000km.

¿Alguno de estos te interesa más?

